In [7]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


In [8]:
import pandas as pd
from datasets import Dataset,DatasetDict
from transformers import AutoTokenizer


train_df=pd.read_csv("train.csv")
val_df=pd.read_csv("val.csv")
test_df=pd.read_csv("test.csv")

unique_labels=sorted(train_df["label"].unique())
label2id={label:i for i, label in enumerate(unique_labels)}
id2label={i: label for i,label in enumerate(unique_labels)}

train_df["label"]=train_df["label"].map(label2id)
val_df["label"]=val_df["label"].map(label2id)
test_df["label"]=test_df["label"].map(label2id)

hf_dataset= DatasetDict({
    "train":Dataset.from_pandas(train_df[["text","label"]]),
    "validation":Dataset.from_pandas(val_df[["text","label"]]),
    "test":Dataset.from_pandas(test_df[["text","label"]])
})


model_name="xlm-roberta-base"
tokenizer=AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
  return tokenizer(examples["text"],padding="max_length",truncation=True,max_length=128)

tokenized_datasets=hf_dataset.map(tokenize_function,batched=True)

print(tokenized_datasets)
print(f"\nKeys in a single tokenized row: {list(tokenized_datasets['train'][0].keys())}")




config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/4493 [00:00<?, ? examples/s]

Map:   0%|          | 0/969 [00:00<?, ? examples/s]

Map:   0%|          | 0/965 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 4493
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 969
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 965
    })
})

Keys in a single tokenized row: ['text', 'label', 'input_ids', 'attention_mask']


In [11]:
!pip install -q transformers[sentencepiece] datasets evaluate accelerate scikit-learn

In [14]:
import numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    AutoTokenizer
)
from sklearn.metrics import accuracy_score, f1_score

model_name = "xlm-roberta-base"

# 1. Load the pre-trained model with a fresh classification head
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

# 2. Define the evaluation metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro")
    }

# 3. Configure the Training hyperparameters
training_args = TrainingArguments(
    output_dir="./xlm-roberta-hinglish-banking",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    report_to="none"
)

# 4. Initialize the Hugging Face Trainer (Updated API)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,        # <--- Changed from 'tokenizer' to 'processing_class'
    compute_metrics=compute_metrics,
)

# 5. Start the fine-tuning process
print("Starting fine-tuning...")
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting fine-tuning...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.660242,0.809082,0.792138
2,1.146765,0.486214,0.859649,0.852084
3,1.146765,0.440432,0.875129,0.868587
4,0.293950,0.439694,0.883385,0.878933


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1124, training_loss=0.6628441844546498, metrics={'train_runtime': 651.9663, 'train_samples_per_second': 27.566, 'train_steps_per_second': 1.724, 'total_flos': 1182264112926720.0, 'train_loss': 0.6628441844546498, 'epoch': 4.0})

In [15]:
print("Evaluating on held-out test set...")
test_results = trainer.evaluate(tokenized_datasets["test"])

print("\n--- Final XLM-RoBERTa Test Metrics ---")
print(f"Test Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"Test Macro-F1: {test_results['eval_macro_f1']:.4f}")
print(f"Test Loss:     {test_results['eval_loss']:.4f}")

Evaluating on held-out test set...


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.293950,0.284226,4,0.931606,0.929211



--- Final XLM-RoBERTa Test Metrics ---
Test Accuracy: 0.9316
Test Macro-F1: 0.9292
Test Loss:     0.2842


In [16]:
# 1. Update model name to MuRIL
model_name = "google/muril-base-cased"

# 2. Load MuRIL's specific tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. Re-tokenize the raw dataset with MuRIL's vocabulary
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

# 4. Load the MuRIL model with a fresh classification head
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

# 5. Configure Training Arguments
training_args = TrainingArguments(
    output_dir="./muril-hinglish-banking",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    report_to="none"
)

# 6. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# 7. Train and instantly evaluate on the Test split
print("Starting MuRIL fine-tuning...")
trainer.train()

print("\nEvaluating MuRIL on held-out test set...")
test_results = trainer.evaluate(tokenized_datasets["test"])

print("\n--- Final MuRIL Test Metrics ---")
print(f"Test Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"Test Macro-F1: {test_results['eval_macro_f1']:.4f}")
print(f"Test Loss:     {test_results['eval_loss']:.4f}")

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Map:   0%|          | 0/4493 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating MuRIL on held-out test set...


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
1.713785,1.552660,4,0.731606,0.643642



--- Final MuRIL Test Metrics ---
Test Accuracy: 0.7316
Test Macro-F1: 0.6436
Test Loss:     1.5527
